In [2]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Using device:", device)


PyTorch version: 2.11.0+cu130
Using device: cuda


In [4]:

# Step 2: Download the model and CIFAR-10 datasets

!pip -q install gdown

# Pretrained neural network
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt

# 10,000 training images
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt

# 1,000 testing images
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt


Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 32.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=353f933b-8e6c-41ab-903f-4bf2299e4a9c
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 35.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 20.6MB/s]


In [5]:

from pathlib import Path

files = [
    "linf_base.pt",
    "cifar10_train_10k.pt",
    "cifar10_test_1k.pt"
]

for filename in files:
    path = Path(filename)
    if path.is_file():
        print(filename, "FOUND")
    else:
        print(filename, "MISSING")


linf_base.pt FOUND
cifar10_train_10k.pt FOUND
cifar10_test_1k.pt FOUND


In [6]:

# Step 3: Load CIFAR-10 datasets

def load_data(path):
    # Load images and labels
    x, y = torch.load(path)

    # Convert image pixels to values between 0 and 1
    # Move images and labels to GPU
    return x.float().div(255).to(device), y.to(device)


# Load training dataset
x_train, y_train = load_data("cifar10_train_10k.pt")

# Load testing dataset
x_test, y_test = load_data("cifar10_test_1k.pt")


# Check the dataset shapes
print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)

print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)

print("Image pixel range:", x_train.min().item(), x_train.max().item())
print("Device:", x_train.device)


Training images: torch.Size([10000, 3, 32, 32])
Training labels: torch.Size([10000])
Testing images: torch.Size([1000, 3, 32, 32])
Testing labels: torch.Size([1000])
Image pixel range: 0.0 1.0
Device: cuda:0


In [9]:

# Step 4: Define the neural network

class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()

        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(
            in_planes, planes, 3, stride, 1, bias=False
        )

        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(
            planes, planes, 3, 1, 1, bias=False
        )

        self.shortcut = None

        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(
                in_planes, planes, 1, stride, bias=False
            )

    def forward(self, x):
        out = F.relu(self.bn1(x))

        shortcut = (
            self.shortcut(out)
            if self.shortcut is not None
            else x
        )

        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))

        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()

        self.register_buffer(
            "mean",
            torch.tensor([0.4914, 0.4822, 0.4465])
            .view(1, 3, 1, 1)
        )

        self.register_buffer(
            "std",
            torch.tensor([0.2471, 0.2435, 0.2616])
            .view(1, 3, 1, 1)
        )

        self.conv1 = nn.Conv2d(
            3, width, 3, 1, 1, bias=False
        )

        layers, in_planes = [], width

        for mult, stride in [
            (1, 1), (1, 1),
            (2, 2), (2, 1),
            (4, 2), (4, 1),
            (8, 2), (8, 1)
        ]:
            layers.append(
                PreActBlock(in_planes, mult * width, stride)
            )
            in_planes = mult * width

        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(
            self.conv1((x - self.mean) / self.std)
        )

        out = F.adaptive_avg_pool2d(
            F.relu(self.bn(out)), 1
        ).flatten(1)

        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(
        torch.load("linf_base.pt", map_location=device)
    )
    return model


In [10]:

# Load the pretrained model

model = load_base_model()
model.eval()

print("Model loaded successfully!")

print(
    "Number of parameters:",
    sum(p.numel() for p in model.parameters())
)

print("Model device:", next(model.parameters()).device)


Model loaded successfully!
Number of parameters: 2796714
Model device: cuda:0


In [12]:

# Step 5A: Helper functions for PGD attacks

def clip_box(x, delta):
    # Keep the adversarial image between 0 and 1
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    # Allow PyTorch to calculate gradients for delta
    delta = delta.detach().requires_grad_(True)

    # Calculate classification loss
    loss = F.cross_entropy(
        model(x + delta), y, reduction="sum"
    )

    # Calculate the gradient of loss with respect to delta
    (grad,) = torch.autograd.grad(loss, delta)

    return grad.detach()


def l2_norm(v):
    # Calculate L2 norm for each image
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


In [13]:

# Step 5B: L-infinity PGD attack

def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):

    # Start with no perturbation
    delta = torch.zeros_like(x)

    # Random initialization inside the epsilon range
    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)

    # Keep image pixel values between 0 and 1
    delta = clip_box(x, delta)

    # Perform PGD attack
    for _ in range(steps):

        # Calculate gradient
        grad = input_grad(model, x, delta, y)

        # Update perturbation
        delta = delta + alpha * grad.sign()

        # Keep perturbation within epsilon
        delta = delta.clamp(-eps, eps)

        # Keep image values between 0 and 1
        delta = clip_box(x, delta)

    # Return adversarial images
    return (x + delta).detach()


In [14]:

# Step 5C: Test L-infinity PGD

model.eval()

x_sample = x_test[:10]
y_sample = y_test[:10]

eps = 8 / 255
alpha = 2 / 255
steps = 10

x_adv = pgd_linf(
    model,
    x_sample,
    y_sample,
    eps,
    alpha,
    steps
)

# Check maximum pixel perturbation
max_change = (x_adv - x_sample).abs().max().item()

# Check adversarial image range
min_pixel = x_adv.min().item()
max_pixel = x_adv.max().item()

# Compare predictions
with torch.no_grad():
    clean_acc = (
        model(x_sample).argmax(1) == y_sample
    ).float().mean().item() * 100

    adv_acc = (
        model(x_adv).argmax(1) == y_sample
    ).float().mean().item() * 100

print("Epsilon:", eps)
print("Maximum pixel change:", max_change)
print("Adversarial pixel range:", min_pixel, max_pixel)
print("Clean accuracy:", clean_acc, "%")
print("PGD accuracy:", adv_acc, "%")


Epsilon: 0.03137254901960784
Maximum pixel change: 0.03137257695198059
Adversarial pixel range: 0.0 1.0
Clean accuracy: 80.0000011920929 %
PGD accuracy: 40.00000059604645 %


In [15]:

# Step 6A: L2 PGD attack

def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):

    delta = torch.zeros_like(x)

    # Random initialization inside the L2 ball
    if rand_init:
        delta = torch.randn_like(x)

        # Normalize the random direction
        delta = delta / l2_norm(delta).clamp_min(1e-12)

        # Random radius inside epsilon
        radius = torch.rand(
            x.shape[0], 1, 1, 1, device=x.device
        )

        delta = delta * eps * radius

    delta = clip_box(x, delta)

    # Perform PGD steps
    for _ in range(steps):

        grad = input_grad(model, x, delta, y)

        # Normalize gradient direction
        grad = grad / l2_norm(grad).clamp_min(1e-12)

        # Update perturbation
        delta = delta + alpha * grad

        # Project back into L2 ball
        norm = l2_norm(delta).clamp_min(1e-12)
        delta = delta * (eps / norm).clamp(max=1.0)

        # Keep image values between 0 and 1
        delta = clip_box(x, delta)

    return (x + delta).detach()


In [16]:

# Step 6B: Test L2 PGD

model.eval()

eps_l2 = 0.5
alpha_l2 = 0.1
steps = 10

x_adv_l2 = pgd_l2(
    model,
    x_sample,
    y_sample,
    eps_l2,
    alpha_l2,
    steps
)

# Calculate L2 perturbation for each image
changes = x_adv_l2 - x_sample
l2_changes = l2_norm(changes)

with torch.no_grad():
    adv_acc_l2 = (
        model(x_adv_l2).argmax(1) == y_sample
    ).float().mean().item() * 100

print("L2 epsilon:", eps_l2)
print("Maximum L2 perturbation:", l2_changes.max().item())
print("Clean accuracy:", round(clean_acc, 2), "%")
print("L2 PGD accuracy:", round(adv_acc_l2, 2), "%")


L2 epsilon: 0.5
Maximum L2 perturbation: 0.5
Clean accuracy: 80.0 %
L2 PGD accuracy: 60.0 %


In [17]:

# Step 7A: Single-norm adversarial training loss

def single_norm_loss(model, x, y, norm, eps, steps=10):

    # Use evaluation mode when generating attacks
    model.eval()

    # Set PGD step size
    alpha = 2.5 * eps / steps

    # Generate adversarial images
    if norm == "linf":
        x_adv = pgd_linf(
            model, x, y, eps, alpha, steps
        )

    elif norm == "l2":
        x_adv = pgd_l2(
            model, x, y, eps, alpha, steps
        )

    else:
        raise ValueError("Unsupported norm")

    # Switch to training mode
    model.train()

    # Calculate loss on adversarial images
    loss = F.cross_entropy(model(x_adv), y)

    return loss


In [18]:

# Step 7B: Test single_norm_loss

# Use a fresh copy of the pretrained model
train_model = load_base_model()

# Take a small batch of training images
x_batch = x_train[:16]
y_batch = y_train[:16]

# Calculate adversarial training loss
loss = single_norm_loss(
    train_model,
    x_batch,
    y_batch,
    norm="linf",
    eps=8/255,
    steps=10
)

print("Adversarial training loss:", loss.item())
print("Model training mode:", train_model.training)


Adversarial training loss: 1.9754867553710938
Model training mode: True


In [19]:

# Step 8A: Data augmentation and batching

def augment(x):
    n = x.shape[0]

    # Random horizontal flip
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(
        flip.view(-1, 1, 1, 1),
        x.flip(3),
        x
    )

    # Random crop with padding
    padded = F.pad(x, (4, 4, 4, 4))

    i = (
        torch.randint(0, 9, (n, 1, 1), device=x.device)
        + torch.arange(32, device=x.device).view(1, 32, 1)
    )

    j = (
        torch.randint(0, 9, (n, 1, 1), device=x.device)
        + torch.arange(32, device=x.device).view(1, 1, 32)
    )

    return padded[
        torch.arange(n, device=x.device).view(n, 1, 1),
        :, i, j
    ].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    if shuffle:
        idx = torch.randperm(len(x), device=x.device)
    else:
        idx = torch.arange(len(x), device=x.device)

    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]


In [20]:

# Step 8B: Adversarial training

def adv_train(
    model,
    loss_fn,
    epochs=10,
    lr=0.01,
    batch_size=128,
    seed=0,
    **kwargs
):
    torch.manual_seed(seed)

    opt = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=5e-4
    )

    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        epochs * (len(x_train) // batch_size)
    )

    start = time.time()

    for epoch in range(epochs):
        losses = []

        for x, y in batches(
            x_train, y_train, batch_size, shuffle=True
        ):
            if len(y) < batch_size:
                continue

            # Apply data augmentation
            x = augment(x)

            # Generate adversarial examples and calculate loss
            loss = loss_fn(model, x, y, **kwargs)

            # Update model weights
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()

            losses.append(loss.item())

        print(
            f"Epoch {epoch + 1}/{epochs}: "
            f"loss {np.mean(losses):.3f} "
            f"({time.time() - start:.0f} s)"
        )

    model.eval()
    return model


In [21]:

# Step 8C: Check training setup

print("Training images:", len(x_train))
print("Batch size:", 128)
print("Full batches per epoch:", len(x_train) // 128)
print("Training epochs:", 10)
print("Training device:", device)
print("Training functions ready!")


Training images: 10000
Batch size: 128
Full batches per epoch: 78
Training epochs: 10
Training device: cuda
Training functions ready!


In [22]:

# Step 9A: Train with L-infinity PGD

eps_train = 8 / 255

# Load a fresh copy of the pretrained model
model_linf_8 = load_base_model()

# Perform adversarial training
model_linf_8 = adv_train(
    model_linf_8,
    single_norm_loss,
    epochs=10,
    lr=0.01,
    batch_size=128,
    seed=0,
    norm="linf",
    eps=eps_train,
    steps=10
)

print("Adversarial training completed!")


Epoch 1/10: loss 1.203 (40 s)
Epoch 2/10: loss 1.181 (80 s)
Epoch 3/10: loss 1.151 (119 s)
Epoch 4/10: loss 1.126 (159 s)
Epoch 5/10: loss 1.096 (199 s)
Epoch 6/10: loss 1.068 (238 s)
Epoch 7/10: loss 1.049 (278 s)
Epoch 8/10: loss 1.030 (318 s)
Epoch 9/10: loss 1.013 (358 s)
Epoch 10/10: loss 1.007 (397 s)
Adversarial training completed!


In [23]:

# Save the trained model

torch.save(
    model_linf_8.state_dict(),
    "CS521_HW3_linf_eps8.pt"
)

print("Model saved successfully!")


Model saved successfully!


In [24]:

# Step 10: Compare standard accuracy

def test_clean_accuracy(model, x, y, batch_size=100):
    model.eval()
    correct = 0

    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            x_batch = x[i:i + batch_size]
            y_batch = y[i:i + batch_size]

            predictions = model(x_batch).argmax(dim=1)
            correct += (predictions == y_batch).sum().item()

    return 100 * correct / len(y)


# Original pretrained model
original_model = load_base_model()

# Evaluate original model
original_acc = test_clean_accuracy(
    original_model, x_test, y_test
)

# Evaluate adversarially trained model
trained_acc = test_clean_accuracy(
    model_linf_8, x_test, y_test
)

print(f"Original model standard accuracy: {original_acc:.2f}%")
print(f"Trained model standard accuracy: {trained_acc:.2f}%")


Original model standard accuracy: 78.70%
Trained model standard accuracy: 78.60%


In [25]:

# Step 11A: Evaluate L-infinity PGD robust accuracy

def test_pgd_accuracy(
    model, x, y, eps=8/255,
    steps=50, restarts=2, batch_size=100
):
    model.eval()

    alpha = 2.5 * eps / steps
    correct = 0

    # Same random starts for fair comparison
    torch.manual_seed(0)

    for i in range(0, len(x), batch_size):
        x_batch = x[i:i + batch_size]
        y_batch = y[i:i + batch_size]

        with torch.no_grad():
            robust = (
                model(x_batch).argmax(1) == y_batch
            )

        for _ in range(restarts):
            x_adv = pgd_linf(
                model, x_batch, y_batch,
                eps, alpha, steps
            )

            with torch.no_grad():
                robust &= (
                    model(x_adv).argmax(1) == y_batch
                )

        correct += robust.sum().item()

    return 100 * correct / len(y)


In [26]:

# Step 11B: Compare PGD robust accuracy

original_robust = test_pgd_accuracy(
    original_model, x_test, y_test
)

trained_robust = test_pgd_accuracy(
    model_linf_8, x_test, y_test
)

print("Original model:")
print(f"  Standard accuracy: {original_acc:.2f}%")
print(f"  PGD robust accuracy: {original_robust:.2f}%")

print("\nAdversarially trained model:")
print(f"  Standard accuracy: {trained_acc:.2f}%")
print(f"  PGD robust accuracy: {trained_robust:.2f}%")

print(
    "\nRobust accuracy difference:",
    f"{trained_robust - original_robust:+.2f} percentage points"
)


Original model:
  Standard accuracy: 78.70%
  PGD robust accuracy: 48.30%

Adversarially trained model:
  Standard accuracy: 78.60%
  PGD robust accuracy: 44.80%

Robust accuracy difference: -3.50 percentage points


In [27]:

# Step 12A: Compare different PGD epsilon values

epsilon_values = [2/255, 4/255, 8/255, 12/255]

pgd_results = []

for eps in epsilon_values:

    print(f"\nTesting epsilon = {eps * 255:.0f}/255")

    original_robust = test_pgd_accuracy(
        original_model,
        x_test,
        y_test,
        eps=eps,
        steps=50,
        restarts=2
    )

    trained_robust = test_pgd_accuracy(
        model_linf_8,
        x_test,
        y_test,
        eps=eps,
        steps=50,
        restarts=2
    )

    pgd_results.append({
        "epsilon": f"{eps * 255:.0f}/255",
        "original": original_robust,
        "trained": trained_robust
    })

    print(f"Original model: {original_robust:.2f}%")
    print(f"Trained model:  {trained_robust:.2f}%")



Testing epsilon = 2/255
Original model: 72.10%
Trained model:  72.00%

Testing epsilon = 4/255
Original model: 64.50%
Trained model:  62.70%

Testing epsilon = 8/255
Original model: 48.30%
Trained model:  44.80%

Testing epsilon = 12/255
Original model: 31.30%
Trained model:  28.90%


In [28]:

# Step 12B: Display PGD comparison results

import pandas as pd

pgd_df = pd.DataFrame(pgd_results)

pgd_df["difference"] = (
    pgd_df["trained"] - pgd_df["original"]
)

print(pgd_df.to_string(index=False))


epsilon  original  trained  difference
  2/255      72.1     72.0        -0.1
  4/255      64.5     62.7        -1.8
  8/255      48.3     44.8        -3.5
 12/255      31.3     28.9        -2.4


In [29]:

# Step 13A: Untargeted FGSM attack

def fgsm_attack(model, x, y, eps):

    # Start with no perturbation
    delta = torch.zeros_like(x)

    # Calculate gradient
    grad = input_grad(model, x, delta, y)

    # One gradient step
    delta = eps * grad.sign()

    # Keep image values between 0 and 1
    delta = clip_box(x, delta)

    return (x + delta).detach()


In [30]:

# Step 13B: Evaluate FGSM accuracy

def test_fgsm_accuracy(model, x, y, eps, batch_size=100):

    model.eval()
    correct = 0

    for i in range(0, len(x), batch_size):

        x_batch = x[i:i + batch_size]
        y_batch = y[i:i + batch_size]

        # Generate FGSM adversarial images
        x_adv = fgsm_attack(
            model, x_batch, y_batch, eps
        )

        with torch.no_grad():
            clean_correct = (
                model(x_batch).argmax(1) == y_batch
            )

            adv_correct = (
                model(x_adv).argmax(1) == y_batch
            )

            # Count images correct before and after attack
            correct += (
                clean_correct & adv_correct
            ).sum().item()

    return 100 * correct / len(y)


In [31]:

# Step 13C: FGSM comparison

epsilon_values = [2/255, 4/255, 8/255, 12/255]

fgsm_results = []

for eps in epsilon_values:

    original_fgsm = test_fgsm_accuracy(
        original_model, x_test, y_test, eps
    )

    trained_fgsm = test_fgsm_accuracy(
        model_linf_8, x_test, y_test, eps
    )

    fgsm_results.append({
        "epsilon": f"{eps * 255:.0f}/255",
        "original": original_fgsm,
        "trained": trained_fgsm
    })

    print(f"\nFGSM epsilon = {eps * 255:.0f}/255")
    print(f"Original model: {original_fgsm:.2f}%")
    print(f"Trained model:  {trained_fgsm:.2f}%")



FGSM epsilon = 2/255
Original model: 72.20%
Trained model:  72.20%

FGSM epsilon = 4/255
Original model: 65.80%
Trained model:  63.90%

FGSM epsilon = 8/255
Original model: 52.50%
Trained model:  49.50%

FGSM epsilon = 12/255
Original model: 41.40%
Trained model:  39.90%


In [32]:

# Step 14: Train with epsilon = 4/255

eps_train_4 = 4 / 255

# Start from the same pretrained model
model_linf_4 = load_base_model()

# Adversarial training
model_linf_4 = adv_train(
    model_linf_4,
    single_norm_loss,
    epochs=10,
    lr=0.01,
    batch_size=128,
    seed=0,
    norm="linf",
    eps=eps_train_4,
    steps=10
)

# Save the model
torch.save(
    model_linf_4.state_dict(),
    "CS521_HW3_linf_eps4.pt"
)

print("Training with epsilon = 4/255 completed!")


Epoch 1/10: loss 0.846 (40 s)
Epoch 2/10: loss 0.800 (80 s)
Epoch 3/10: loss 0.760 (120 s)
Epoch 4/10: loss 0.723 (159 s)
Epoch 5/10: loss 0.686 (199 s)
Epoch 6/10: loss 0.650 (239 s)
Epoch 7/10: loss 0.619 (278 s)
Epoch 8/10: loss 0.595 (318 s)
Epoch 9/10: loss 0.572 (358 s)
Epoch 10/10: loss 0.564 (398 s)
Training with epsilon = 4/255 completed!


In [33]:

# Step 15A: Evaluate model trained with epsilon = 4/255

# Standard accuracy
clean_acc_4 = test_clean_accuracy(
    model_linf_4,
    x_test,
    y_test
)

# PGD robust accuracy at epsilon = 8/255
robust_acc_4 = test_pgd_accuracy(
    model_linf_4,
    x_test,
    y_test,
    eps=8/255,
    steps=50,
    restarts=2
)

print("Model trained with epsilon = 4/255")
print(f"Standard accuracy: {clean_acc_4:.2f}%")
print(f"PGD robust accuracy at 8/255: {robust_acc_4:.2f}%")


Model trained with epsilon = 4/255
Standard accuracy: 83.20%
PGD robust accuracy at 8/255: 36.30%


In [34]:

# Step 15B: Compare training epsilon values

import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Original pretrained",
        "Trained eps = 4/255",
        "Trained eps = 8/255"
    ],
    "Standard Accuracy (%)": [
        original_acc,
        clean_acc_4,
        trained_acc
    ],
    "PGD Robust Accuracy (%)": [
        48.30,
        robust_acc_4,
        44.80
    ]
})

print(comparison.to_string(index=False))


              Model  Standard Accuracy (%)  PGD Robust Accuracy (%)
Original pretrained                   78.7                     48.3
Trained eps = 4/255                   83.2                     36.3
Trained eps = 8/255                   78.6                     44.8


In [35]:

# Step 16: Train model with epsilon = 2/255

eps_train_2 = 2 / 255

# Start from the same pretrained model
model_linf_2 = load_base_model()

# Perform adversarial training
model_linf_2 = adv_train(
    model_linf_2,
    single_norm_loss,
    epochs=10,
    lr=0.01,
    batch_size=128,
    seed=0,
    norm="linf",
    eps=eps_train_2,
    steps=10
)

# Save trained model
torch.save(
    model_linf_2.state_dict(),
    "CS521_HW3_linf_eps2.pt"
)

print("Training with epsilon = 2/255 completed!")


Epoch 1/10: loss 0.680 (40 s)
Epoch 2/10: loss 0.599 (80 s)
Epoch 3/10: loss 0.546 (119 s)
Epoch 4/10: loss 0.507 (159 s)
Epoch 5/10: loss 0.462 (199 s)
Epoch 6/10: loss 0.420 (239 s)
Epoch 7/10: loss 0.385 (278 s)
Epoch 8/10: loss 0.358 (318 s)
Epoch 9/10: loss 0.335 (358 s)
Epoch 10/10: loss 0.325 (397 s)
Training with epsilon = 2/255 completed!


In [36]:

# Step 17A: Evaluate model trained with epsilon = 2/255

clean_acc_2 = test_clean_accuracy(
    model_linf_2,
    x_test,
    y_test
)

robust_acc_2 = test_pgd_accuracy(
    model_linf_2,
    x_test,
    y_test,
    eps=8/255,
    steps=50,
    restarts=2
)

print("Model trained with epsilon = 2/255")
print(f"Standard accuracy: {clean_acc_2:.2f}%")
print(f"PGD robust accuracy: {robust_acc_2:.2f}%")


Model trained with epsilon = 2/255
Standard accuracy: 85.20%
PGD robust accuracy: 27.60%


In [37]:

# Step 17B: Final training epsilon comparison

import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Original pretrained",
        "Trained eps = 2/255",
        "Trained eps = 4/255",
        "Trained eps = 8/255"
    ],
    "Standard Accuracy (%)": [
        original_acc,
        clean_acc_2,
        clean_acc_4,
        trained_acc
    ],
    "PGD Robust Accuracy (%)": [
        48.30,
        robust_acc_2,
        robust_acc_4,
        44.80
    ]
})

print(comparison.to_string(index=False))


              Model  Standard Accuracy (%)  PGD Robust Accuracy (%)
Original pretrained                   78.7                     48.3
Trained eps = 2/255                   85.2                     27.6
Trained eps = 4/255                   83.2                     36.3
Trained eps = 8/255                   78.6                     44.8


In [38]:

# Step 18: Compare PGD and FGSM across trained models

models = {
    "Original": original_model,
    "Train 2/255": model_linf_2,
    "Train 4/255": model_linf_4,
    "Train 8/255": model_linf_8
}

attack_epsilons = [2/255, 4/255, 8/255, 12/255]

results = []

for name, current_model in models.items():

    for eps in attack_epsilons:

        pgd_acc = test_pgd_accuracy(
            current_model,
            x_test,
            y_test,
            eps=eps,
            steps=50,
            restarts=2
        )

        fgsm_acc = test_fgsm_accuracy(
            current_model,
            x_test,
            y_test,
            eps=eps
        )

        results.append({
            "Model": name,
            "Attack epsilon": f"{eps * 255:.0f}/255",
            "PGD accuracy": pgd_acc,
            "FGSM accuracy": fgsm_acc
        })

import pandas as pd

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))


      Model Attack epsilon  PGD accuracy  FGSM accuracy
   Original          2/255          72.1           72.2
   Original          4/255          64.5           65.8
   Original          8/255          48.3           52.5
   Original         12/255          31.3           41.4
Train 2/255          2/255          72.5           73.4
Train 2/255          4/255          57.2           60.7
Train 2/255          8/255          27.6           41.3
Train 2/255         12/255          10.2           27.5
Train 4/255          2/255          72.1           72.7
Train 4/255          4/255          60.8           63.2
Train 4/255          8/255          36.3           45.9
Train 4/255         12/255          17.7           32.4
Train 8/255          2/255          72.0           72.2
Train 8/255          4/255          62.7           63.9
Train 8/255          8/255          44.8           49.5
Train 8/255         12/255          28.9           39.9
